# Goal task 9 on Colab, notebook 1 of 2
1. Runtime → Change runtime type → **G4** (96 GB) or H100. Not A100, L4 or T4: too small for the 54 GB teacher.
2. Key icon (left) → add a secret **`HF_TOKEN`** → turn on notebook access.
3. Runtime → **Run all**, and allow Google Drive access.

This notebook runs 12 of the 24 trials (shard 0/2). Keep the tab open. After a disconnect, Run all again: it resumes.

In [ ]:
SHARD = '0/2'  # this notebook's half of the 24 trials
import os, shutil, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # the key icon on the left: a secret named HF_TOKEN, notebook access on
os.environ['SHARD'] = SHARD
gpu, mem = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True).split(',')
ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
disk = shutil.disk_usage('/content').free / 2**30
print(f'GPU {gpu} ({int(mem) // 1024} GB), {os.cpu_count()} CPUs, {ram:.0f} GB RAM, {disk:.0f} GB free disk')
assert int(mem) > 70000, 'the teacher needs a GPU of 80 GB or more: Runtime > Change runtime type > G4 (or H100)'
assert os.cpu_count() >= 6, 'too few CPUs for the simulators'
assert disk > 90, 'too little disk for the environments (17 GB) and the teacher (52 GB)'
if ram < 80:
    print(f'warning: {ram:.0f} GB RAM; the run used up to 75 GB on O2')
print('ok: run the next cell')


In [ ]:
# Runs everything: setup (1-2 h the first time), the teacher's weights, the episodes, the upload. Progress shows below:
# each step (==), a heartbeat every 2 min with the latest log line (..), and a bar of finished episodes. After a
# disconnect, Run all again: it resumes where it stopped.
import os, re, subprocess, time
from tqdm.auto import tqdm
URL = 'https://raw.githubusercontent.com/ericjkge/robot-distillation/claude/on-policy-distillation/research/experiments/on-policy-distillation/colab/goal9.sh'
subprocess.run(['curl', '-sSfL', URL, '-o', '/content/goal9.sh'], check=True)
bar = tqdm(total=12, desc='episodes', unit='episode', bar_format='{desc}: {n}/{total} |{bar}| {elapsed} elapsed{postfix}')
proc = subprocess.Popen(['bash', '/content/goal9.sh'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
kept, last, refreshed = 0, [], time.time()
for line in proc.stdout:
    line = line.rstrip()
    last = (last + [line])[-40:]
    if line.startswith(('== ', '.. ', 'done:')):
        bar.write(line)
        if m := re.search(r'running (\d+) episodes', line):
            bar.reset(total=int(m[1]))
    elif 'EPISODE_END' in line:
        kept += "'success': True" in line
        bar.update(1)
        bar.set_postfix_str(f'{kept} kept')
    elif line.startswith('Traceback') or re.match(r'\w*Error\b', line):
        bar.write(line[:300])
    if time.time() - refreshed > 30:
        bar.refresh()
        refreshed = time.time()
proc.wait()
bar.close()
if proc.returncode:
    print('\n'.join(last))
    raise RuntimeError(f'stopped with an error (exit {proc.returncode}): paste the lines above to Claude')
